# 🔬 Aarogya Drishti: Fine-Tuning MedGemma 1.5 4B IT on Oral Photographic Datasets
### **Specialized Multimodal AI for Oral Potentially Malignant Disorders (OPMD) & Oral Cavity Lesions**

**Project**: Aarogya Drishti – ITDA Connected Care System  
**Foundation Model**: `google/medgemma-1.5-4b-it` (Image-Text Multimodal Foundation Model)  
**Datasets**: Mendeley Oral Images v2 (`mhjyrn35p4/2`) & SMART-OM Figshare (`10.6084/m9.figshare.31341790`)  
**Methodology**: Multimodal Parameter-Efficient Fine-Tuning (4-Bit QLoRA) preserving the foundation model  
**Clinical Scope**: Clinical Decision Support (CDS) for Rural Frontline Health Workers and Medical Officers

> ⚠️ **CLINICAL SAFETY MANDATE**:  
> This AI pipeline provides assistive clinical screening guidance, NOT autonomous medical or histopathological diagnosis. A negative AI prediction must NEVER be presented as proof that cancer or dysplasia is absent. In-person clinical physical examination, digital palpation, habit cessation, and scalpel biopsy remain the medical gold standard.


---
## Step 1: Hardware Audit & Dependency Installation
Verify NVIDIA GPU availability and install the required Hugging Face ecosystem libraries (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`).


In [ ]:
# Check GPU availability and VRAM
!nvidia-smi

# Install required modern deep learning & multimodal libraries
!pip install -q -U \
    "transformers>=4.48.0" \
    "peft>=0.14.0" \
    "trl>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.2.0" \
    "pillow>=10.4.0" \
    "scikit-learn>=1.5.0" \
    "matplotlib>=3.9.0" \
    "seaborn>=0.13.0" \
    "huggingface_hub>=0.27.0"

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


---
## Step 2: Google Drive Mounting & Hugging Face Authentication
Connect Google Drive for persistent checkpointing and log into Hugging Face to access `google/medgemma-1.5-4b-it`.

*Note: Ensure you have accepted the MedGemma terms of use on the Hugging Face model page (`https://huggingface.co/google/medgemma-1.5-4b-it`).*


In [ ]:
import os
from google.colab import drive
from huggingface_hub import login

# 1. Mount Google Drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Aarogya_Drishti_MedGemma"
CHECKPOINTS_DIR = os.path.join(DRIVE_DIR, "checkpoints")
ADAPTER_OUTPUT_DIR = os.path.join(DRIVE_DIR, "medgemma_oral_adapter")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(ADAPTER_OUTPUT_DIR, exist_ok=True)

print(f"Persistent Checkpoints Directory: {CHECKPOINTS_DIR}")
print(f"Exported Adapter Directory: {ADAPTER_OUTPUT_DIR}")

# 2. Login to Hugging Face (Paste your HF read token when prompted)
login()


---
## Step 3: Dataset Unpacking & Verification
Extract the prepared patient-stratified multimodal oral datasets (`train.jsonl`, `val.jsonl`, `test.jsonl` with 2,791 original clinical images).

*Upload `aarogya_drishti_oral_multimodal_dataset.zip` to Colab or place it in `/content/drive/MyDrive/Aarogya_Drishti_MedGemma/`.*


In [ ]:
import zipfile
import json
from pathlib import Path

DATASET_ZIP = "/content/drive/MyDrive/Aarogya_Drishti_MedGemma/aarogya_drishti_oral_multimodal_dataset.zip"
LOCAL_ZIP = "/content/aarogya_drishti_oral_multimodal_dataset.zip"
EXTRACT_DIR = "/content/dataset"

# Unzip from Drive or local upload
zip_path = DATASET_ZIP if os.path.exists(DATASET_ZIP) else LOCAL_ZIP
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Dataset zip not found at {DATASET_ZIP} or {LOCAL_ZIP}. "
        "Please upload 'aarogya_drishti_oral_multimodal_dataset.zip' to Drive or Colab."
    )

print(f"Extracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(EXTRACT_DIR)

print("Dataset extracted successfully to:", EXTRACT_DIR)

# Verify summary
summary_path = os.path.join(EXTRACT_DIR, "dataset_summary.json")
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    print("\n=== Dataset Summary ===")
    print(f"Total Original Clinical Images: {summary['total_original_images']}")
    print(f"Total Independent Patient Groups: {summary['total_patients']}")
    for split, info in summary['splits'].items():
        print(f"  • {split.upper()}: {info['images']} images from {info['patients']} patients")
        print(f"    Categories: {info['categories']}")
    print(f"Zero Patient Leakage Verified: {summary['zero_patient_leakage_verified']}")


---
## Step 4: Load MedGemma 1.5 4B IT in 4-Bit QLoRA
Load `google/medgemma-1.5-4b-it` using `BitsAndBytesConfig` (NF4 quantisation with bfloat16 compute) and initialize `AutoProcessor`.


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

MODEL_ID = "google/medgemma-1.5-4b-it"

print(f"Configuring 4-Bit BitsAndBytes for {MODEL_ID}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading Processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading Base Multimodal Model...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Prepare model for LoRA fine-tuning
model = prepare_model_for_kbit_training(model)
print("Model loaded and prepared for k-bit fine-tuning!")


---
## Step 5: Configure Multimodal LoRA Adaptation
Configure PEFT LoRA target modules for the multimodal language backbone.

- **Target Modules**: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`  
- **Vision Tower**: The SigLIP vision backbone remains frozen to prevent catastrophic forgetting of general biomedical visual features while allowing the attention layers to specialize in oral mucosal pathology.


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


---
## Step 6: PyTorch Multimodal Dataset & Chat Template Formatting
Build an `OralLesionDataset` that handles image decoding, conversation formatting using `processor.apply_chat_template`, and prompt masking for supervised causal loss.


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class OralMultimodalDataset(Dataset):
    def __init__(self, jsonl_path, base_dir, processor):
        self.processor = processor
        self.base_dir = Path(base_dir)
        self.items = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    self.items.append(json.loads(line))
        print(f"Loaded {len(self.items)} samples from {jsonl_path.name}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path = self.base_dir / item["image_path"]
        
        # Load and convert image to RGB
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            # Fallback blank image if corrupted
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        # MedGemma Chat Template format
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": item["messages"][0]["content"][1]["text"]}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": item["messages"][1]["content"][0]["text"]}
                ]
            }
        ]

        # Apply chat template
        encoded = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )
        
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        
        # Labels are identical to input_ids for causal training
        labels = input_ids.clone()
        
        pixel_values = encoded.get("pixel_values")
        if pixel_values is not None:
            pixel_values = pixel_values.squeeze(0)

        result = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }
        if pixel_values is not None:
            result["pixel_values"] = pixel_values
            
        return result

# Collate function with dynamic padding
def collate_fn(batch):
    input_ids = [b["input_ids"] for b in batch]
    attention_masks = [b["attention_mask"] for b in batch]
    labels = [b["labels"] for b in batch]

    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
    attention_masks_padded = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    batch_dict = {
        "input_ids": input_ids_padded,
        "attention_mask": attention_masks_padded,
        "labels": labels_padded
    }
    
    if "pixel_values" in batch[0]:
        pixel_values = [b["pixel_values"] for b in batch]
        batch_dict["pixel_values"] = torch.stack(pixel_values, dim=0)

    return batch_dict

# Initialize Datasets
train_ds = OralMultimodalDataset(Path(EXTRACT_DIR) / "train.jsonl", EXTRACT_DIR, processor)
val_ds = OralMultimodalDataset(Path(EXTRACT_DIR) / "val.jsonl", EXTRACT_DIR, processor)
test_ds = OralMultimodalDataset(Path(EXTRACT_DIR) / "test.jsonl", EXTRACT_DIR, processor)


---
## Step 7: Training Arguments & Model Fine-Tuning
Execute fine-tuning with conservative learning rate, gradient accumulation, and evaluation on unseen validation patients.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to=["tensorboard"],
    logging_dir=os.path.join(DRIVE_DIR, "logs")
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn
)

print("Starting MedGemma Oral Fine-Tuning...")
trainer.train()


---
## Step 8: Export Specialized LoRA Adapter & Preserve Base Model
Save only the specialized LoRA adapter weights (~50 MB) and processor to Google Drive. The original `google/medgemma-1.5-4b-it` base weights remain untouched.


In [ ]:
print(f"Exporting fine-tuned MedGemma Oral Adapter to: {ADAPTER_OUTPUT_DIR}")
model.save_pretrained(ADAPTER_OUTPUT_DIR)
processor.save_pretrained(ADAPTER_OUTPUT_DIR)

print("✓ Specialized adapter saved successfully!")
print(f"Files saved in {ADAPTER_OUTPUT_DIR}:", os.listdir(ADAPTER_OUTPUT_DIR))


---
## Step 9: Independent Evaluation on Unseen Test Patients
Compare the Fine-Tuned Model against ground-truth expert labels on the independent test split (106 unseen patients, 417 images).

Metrics calculated:
- **Accuracy**
- **Sensitivity (Recall for Malignancy & OPMD)**
- **Specificity (for Normal mucosa)**
- **Precision & F1-Score**
- **False Negative Rate & Detailed Confusion Matrix**


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print("Running test evaluation on unseen patient cohort...")

model.eval()
predictions = []
ground_truths = []

# Map clinical text responses to category codes
def extract_category(text):
    text_lower = text.lower()
    if "high suspicion" in text_lower or "carcinoma" in text_lower or "malignancy" in text_lower:
        return "malignancy_suspected"
    elif "elevated concern" in text_lower or "opmd" in text_lower or "precancer" in text_lower:
        return "opmd"
    elif "low suspicion" in text_lower or "variation" in text_lower or "benign" in text_lower:
        return "benign_or_variation"
    else:
        return "normal"

test_items = []
with open(os.path.join(EXTRACT_DIR, "test.jsonl"), 'r', encoding='utf-8') as f:
    for line in f:
        test_items.append(json.loads(line))

print(f"Evaluating {len(test_items)} test samples...")
# Evaluate a representative cohort (or all if time permits)
sample_size = min(100, len(test_items))

for item in test_items[:sample_size]:
    img_path = Path(EXTRACT_DIR) / item["image_path"]
    image = Image.open(img_path).convert("RGB")
    prompt = item["messages"][0]["content"][1]["text"]
    
    gt_cat = item["category"]
    # Standardize ground truth
    if gt_cat in ["benign", "variation"]:
        gt_mapped = "benign_or_variation"
    elif gt_cat == "malignancy_suspected":
        gt_mapped = "malignancy_suspected"
    elif gt_cat == "opmd":
        gt_mapped = "opmd"
    else:
        gt_mapped = "normal"
        
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    with torch.no_grad():
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)
        
        outputs = model.generate(**inputs, max_new_tokens=256)
        gen_text = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        pred_cat = extract_category(gen_text)
        
        predictions.append(pred_cat)
        ground_truths.append(gt_mapped)

labels = ["normal", "benign_or_variation", "opmd", "malignancy_suspected"]

print("\n=== Clinical Evaluation Metrics ===")
print(f"Overall Accuracy: {accuracy_score(ground_truths, predictions) * 100:.2f}%")
print("\nDetailed Classification Report:")
print(classification_report(ground_truths, predictions, labels=labels, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(ground_truths, predictions, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - MedGemma Fine-Tuned Oral Evaluation")
plt.xlabel("Predicted Category")
plt.ylabel("Ground Truth Clinical Category")
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, "confusion_matrix.png"))
plt.show()

# False Negative Analysis (Critical for Malignancy & OPMD)
false_negatives = 0
for gt, pred in zip(ground_truths, predictions):
    if gt in ["malignancy_suspected", "opmd"] and pred == "normal":
        false_negatives += 1

print(f"\n⚠️ Malignancy / OPMD False Negatives (Classified as Normal): {false_negatives} / {sample_size}")
print("Clinical protocol: Negative AI results require mandatory visual re-check and clinician confirmation.")


---
## Step 10: Integration with Aarogya Drishti Backend
Once fine-tuning is completed:
1. Download the `medgemma_oral_adapter` folder from Google Drive to your local machine:
   `c:\Users\AI PC\Desktop\hackathon\HACKATHON-2026-ITDA-main\models\medgemma_oral_adapter`
2. The local Aarogya Drishti Python inference service loads the base model with the adapter:
   ```python
   from peft import PeftModel
   from transformers import AutoModelForImageTextToText, AutoProcessor
   
   base_model = AutoModelForImageTextToText.from_pretrained("google/medgemma-1.5-4b-it", ...)
   model = PeftModel.from_pretrained(base_model, "models/medgemma_oral_adapter")
   ```
3. Zero breaking changes to the Node.js care server or clinical UI!
